## 1. Setup
Mounts Drive, copies the data locally, loads the libraries and holds all settings.

**Tip:** set `QUICK = True` first (5 traces per dataset, takes a minute or two) to check that everything works, then set it to `False` for the real run. Quick results are saved with a `_quick` name and never mixed with the real ones.

**You should see:** the three trace counts and the output folder.

In [7]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, shutil, time, warnings
import numpy as np, pandas as pd
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
from statsmodels.tsa.arima.model import ARIMA


QUICK    = False
ROOT     = "/content/drive/MyDrive/dtn_project"
RES_A    = f"{ROOT}/results"
RES_B    = f"{ROOT}/results_b"
LOCAL    = "/content/processed"
WINDOW, DAY = 12, 288
DATASETS = ["5g", "4g", "operator"]
COLSET   = {"5g": "single", "4g": "single", "operator": "value+daily"}
ORDERS   = [(1, 0, 0), (2, 0, 0), (1, 1, 0), (1, 1, 1), (2, 1, 1)]   # candidate (p, d, q), chosen on VALIDATION
SEL_TRACES = 15          # traces per dataset used to choose the order
MAX_FIT  = 20000         # fit on at most the last 20000 training steps of a trace
N_JOBS   = os.cpu_count() or 1

SUF = "_quick" if QUICK else ""
if QUICK: ORDERS, SEL_TRACES = ORDERS[:3], 4
os.makedirs(RES_B, exist_ok=True)
if not os.path.exists(LOCAL): shutil.copytree(f"{ROOT}/processed", LOCAL)
for ds in DATASETS: print(ds, len(glob.glob(f"{LOCAL}/{ds}/*.npz")), "traces")
print("CPU cores used:", N_JOBS, "| outputs in:", RES_B, "| QUICK =", QUICK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
5g 79 traces
4g 134 traces
operator 55 traces
CPU cores used: 2 | outputs in: /content/drive/MyDrive/dtn_project/results_b | QUICK = False


## 2. Helpers
Same data loading and `metrics` as the first notebook (metrics copied exactly from Member A).

`one_step` is the ARIMA part: it fits on the training data, then runs the fixed model through the series and returns the one-step-ahead prediction for every step from `start` onward.

Nothing is computed here.

In [8]:
def load(ds, limit=None):
    d = f"{LOCAL}/{ds}"
    sc = json.load(open(f"{d}/scaler.json"))
    files = sorted(glob.glob(d + "/*.npz"))
    if limit: files = files[::max(1, len(files) // limit)][:limit]
    out = {}
    for f in files:
        z = np.load(f)
        out[os.path.basename(f)[:-4]] = (z["train"][:, 0].astype("float64"), z["val"][:, 0].astype("float64"),
                                         z["test"][:, 0].astype("float64"))
    return out, sc["mean"][0], sc["std"][0]

def metrics(y, p, last):
    """Same as Member A: all in original units. last = the previous true value (to judge direction)."""
    y, p, last = (np.asarray(a, dtype="float64") for a in (y, p, last))
    err = p - y
    ss = ((y - y.mean()) ** 2).sum()
    ch = y - last; m = ch != 0
    return dict(MAE=np.abs(err).mean(), RMSE=np.sqrt((err ** 2).mean()),
                R2=1 - (err ** 2).sum() / ss if ss > 0 else np.nan,
                dir_acc=(np.sign(p - last)[m] == np.sign(ch)[m]).mean() if m.any() else np.nan)

def one_step(tr, series, order, start):
    """Fit ARIMA on the training part only, keep the parameters fixed, and return the one-step-ahead
    predictions for series[start:], plus the fitting time in seconds."""
    t0 = time.time()
    res = ARIMA(tr[-MAX_FIT:], order=order, trend="c" if order[1] == 0 else "n").fit()
    fit_s = time.time() - t0
    pred = res.apply(series, refit=False).predict(start=start, end=len(series) - 1)
    return np.asarray(pred, dtype="float64"), fit_s

def fit_trace(name, tr, va, te, order):
    """One trace, final run: returns the unscaled arrays, or None if ARIMA failed (then we fall back)."""
    full = np.concatenate([tr, va, te]); n0 = len(tr) + len(va)
    try:
        p, fit_s = one_step(tr, full, order, n0)
        if len(p) != len(te) or not np.all(np.isfinite(p)): raise ValueError("bad prediction")
        return name, p, fit_s, 0
    except Exception:
        return name, full[n0 - 1:-1], 0.0, 1

## 3. Choose the ARIMA order on validation data
Tries each candidate order on a sample of traces: fit on the training part, predict the **validation** part one step ahead, and measure the RMSE. The order with the lowest mean validation RMSE is kept for that dataset. The test data is never used here.

Every try is saved to `arima_order_search.csv` as it finishes, so if Colab disconnects you can run the cell again and it continues.

**You should see:** one line per try, then the chosen order for each dataset.

In [9]:
ORD_CSV = f"{RES_B}/arima_order_search{SUF}.csv"
done = pd.read_csv(ORD_CSV) if os.path.exists(ORD_CSV) else pd.DataFrame(columns=["dataset", "order", "val_RMSE"])

def val_rmse(tr, va, order):
    try:
        p, _ = one_step(tr, np.concatenate([tr, va]), order, len(tr))
        return float(np.sqrt(np.mean((p - va) ** 2))) if np.all(np.isfinite(p)) else np.nan
    except Exception:
        return np.nan

CHOSEN = {}
for ds in DATASETS:
    traces, mu0, sd0 = load(ds)
    names = sorted(traces); rng = np.random.RandomState(0)
    sel = [names[i] for i in sorted(rng.choice(len(names), min(SEL_TRACES, len(names)), replace=False))]
    for o in ORDERS:
        if ((done.dataset == ds) & (done.order == str(o))).any(): continue
        t0 = time.time()
        r = Parallel(n_jobs=N_JOBS)(delayed(val_rmse)(traces[k][0], traces[k][1], o) for k in sel)
        row = dict(dataset=ds, order=str(o), val_RMSE=float(np.nanmean(r)) * sd0)   # in original units
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(ORD_CSV, index=False)
        print(f"{ds} order {o}: validation RMSE {row['val_RMSE']:.3f} ({time.time() - t0:.0f}s)")
    d = done[done.dataset == ds].sort_values("val_RMSE")
    CHOSEN[ds] = eval(d.iloc[0].order)
print("\nChosen orders:", CHOSEN)

5g order (1, 0, 0): validation RMSE 6741.647 (5s)
5g order (2, 0, 0): validation RMSE 6585.537 (7s)
5g order (1, 1, 0): validation RMSE 7226.497 (2s)
5g order (1, 1, 1): validation RMSE 6855.924 (9s)
5g order (2, 1, 1): validation RMSE 6480.067 (13s)
4g order (1, 0, 0): validation RMSE 3663.905 (2s)
4g order (2, 0, 0): validation RMSE 3610.714 (6s)
4g order (1, 1, 0): validation RMSE 3605.424 (2s)
4g order (1, 1, 1): validation RMSE 3517.391 (3s)
4g order (2, 1, 1): validation RMSE 3527.057 (4s)
operator order (1, 0, 0): validation RMSE 8.361 (18s)
operator order (2, 0, 0): validation RMSE 8.227 (27s)
operator order (1, 1, 0): validation RMSE 7.851 (8s)
operator order (1, 1, 1): validation RMSE 7.540 (26s)
operator order (2, 1, 1): validation RMSE 7.516 (41s)

Chosen orders: {'5g': (2, 1, 1), '4g': (1, 1, 1), 'operator': (2, 1, 1)}


## 4. Final ARIMA run on every trace
Fits one ARIMA per test trace (training part only, chosen order, fixed parameters) and predicts the test period one step at a time. Saves:
- `arima_trace_results.csv`: one row per trace (same layout as before, plus `fit_s` = fitting time and `fallback` = 1 if ARIMA failed on that trace).
- `arima_runs.csv`: one row per dataset, with the **total fitting time** over all traces (the cost of ARIMA, to be compared with the neural models' training time later).

Finished traces are skipped on a re-run, so if Colab disconnects, just run this cell again.

**You should see:** a progress line every 10 traces, and per dataset a summary line.

In [10]:
TR_CSV, RUN_CSV = f"{RES_B}/arima_trace_results{SUF}.csv", f"{RES_B}/arima_runs{SUF}.csv"
res = pd.read_csv(TR_CSV) if os.path.exists(TR_CSV) else pd.DataFrame()
runs = []
for ds in DATASETS:
    order = CHOSEN[ds]; rid = f"{ds}|{COLSET[ds]}|w{WINDOW}|arima|s0"
    traces, mu0, sd0 = load(ds, limit=5 if QUICK else None)
    todo = [k for k in traces if not (len(res) and ((res.run_id == rid) & (res.trace == k)).any())]
    print(f"{ds}: order {order}, {len(todo)} traces to do")
    for i in range(0, len(todo), 10):
        chunk = todo[i:i + 10]; t0 = time.time()
        out = Parallel(n_jobs=N_JOBS)(delayed(fit_trace)(k, *traces[k], order) for k in chunk)
        rows = []
        for k, p, fit_s, fb in out:
            tr, va, te = traces[k]; full = np.concatenate([tr, va, te]); idx = len(tr) + len(va) + np.arange(len(te))
            rows.append(dict(trace=k, n=len(te), run_id=rid,
                             **metrics(full[idx] * sd0 + mu0, p * sd0 + mu0, full[idx - 1] * sd0 + mu0),
                             fit_s=fit_s, fallback=fb))
        res = pd.concat([res, pd.DataFrame(rows)], ignore_index=True)
        res.to_csv(TR_CSV, index=False)
        print(f"  {min(i + 10, len(todo))}/{len(todo)} traces ({time.time() - t0:.0f}s for this chunk)")
    r = res[res.run_id == rid]
    runs.append(dict(run_id=rid, dataset=ds, colset=COLSET[ds], window=WINDOW, model="arima", seed=0,
                     hidden=0, lr=0, params=order[0] + order[2] + (1 if order[1] == 0 else 0), epochs=0,
                     best_val=np.nan, train_s=round(r.fit_s.sum(), 1)))
    print(f"{rid}: mean RMSE {r.RMSE.mean():.3f} | median R2 {r.R2.median():.2f} | "
          f"fallbacks {int(r.fallback.sum())} | total fit time {r.fit_s.sum():.0f}s")
pd.DataFrame(runs).to_csv(RUN_CSV, index=False)
print("saved to", RES_B)

5g: order (2, 1, 1), 79 traces to do
  10/79 traces (5s for this chunk)
  20/79 traces (5s for this chunk)
  30/79 traces (12s for this chunk)
  40/79 traces (6s for this chunk)
  50/79 traces (4s for this chunk)
  60/79 traces (5s for this chunk)
  70/79 traces (9s for this chunk)
  79/79 traces (15s for this chunk)
5g|single|w12|arima|s0: mean RMSE 8605.075 | median R2 0.20 | fallbacks 0 | total fit time 115s
4g: order (1, 1, 1), 134 traces to do
  10/134 traces (1s for this chunk)
  20/134 traces (2s for this chunk)
  30/134 traces (2s for this chunk)
  40/134 traces (2s for this chunk)
  50/134 traces (2s for this chunk)
  60/134 traces (3s for this chunk)
  70/134 traces (3s for this chunk)
  80/134 traces (2s for this chunk)
  90/134 traces (1s for this chunk)
  100/134 traces (1s for this chunk)
  110/134 traces (2s for this chunk)
  120/134 traces (2s for this chunk)
  130/134 traces (1s for this chunk)
  134/134 traces (1s for this chunk)
4g|single|w12|arima|s0: mean RMSE 3287

## 5. Check and compare
- Same trace set as the neural models and the other baselines.
- **Fallback count** should be zero or very small. If many traces fell back, ARIMA failed there and its score is really "last value" for them, which must be reported.
- A table with **ARIMA, last value and the neural models side by side** (mean over traces).

**You should see:** `same set: True` for every dataset, and ARIMA close to last value (it is a one-step predictor of the same kind).

In [11]:
def tag(df):
    p = df.run_id.str.split("|")
    df["dataset"], df["colset"], df["window"], df["model"] = p.str[0], p.str[1], p.str[2], p.str[3]
    return df
na = tag(pd.read_csv(f"{RES_A}/neural_trace_results.csv"))
nb = tag(pd.read_csv(f"{RES_B}/baseline_trace_results.csv"))
ar = tag(pd.read_csv(TR_CSV))
for ds in DATASETS:
    a = na[(na.dataset == ds) & (na.colset == COLSET[ds]) & (na.window == f"w{WINDOW}")]
    b = ar[ar.dataset == ds]
    print(f"{ds}: traces neural {a.trace.nunique()} | arima {b.trace.nunique()} | same set: "
          f"{set(a.trace) == set(b.trace) if not QUICK else 'skipped in QUICK'} | fallbacks: {int(b.fallback.sum())}")
both = pd.concat([na[(na.colset == na.dataset.map(COLSET)) & (na.window == f"w{WINDOW}")], nb, ar])
if QUICK: both = both[both.trace.isin(ar.trace)]
tab = both.groupby(["dataset", "model"]).agg(RMSE=("RMSE", "mean"), MAE=("MAE", "mean"),
                                              R2_median=("R2", "median"), dir_acc=("dir_acc", "mean")).round(3)
print(); print(tab.to_string())

5g: traces neural 79 | arima 79 | same set: True | fallbacks: 0
4g: traces neural 134 | arima 134 | same set: True | fallbacks: 0
operator: traces neural 55 | arima 55 | same set: True | fallbacks: 0

                             RMSE       MAE  R2_median  dir_acc
dataset  model                                                 
4g       arima           3287.851  2233.071      0.622    0.600
         bilstm          3294.491  2248.928      0.615    0.579
         conv1d          3277.317  2235.066      0.632    0.586
         gru             3279.344  2232.588      0.621    0.580
         last_value      3429.096  2268.447      0.593    0.000
         lstm            3286.527  2244.676      0.614    0.577
5g       arima           8605.075  4796.878      0.196    0.620
         bilstm          7843.485  4310.465      0.090    0.624
         conv1d          8096.221  4727.087      0.018    0.606
         gru             7900.200  4509.970      0.029    0.624
         last_value      9352.8